# 01 · Limpeza e preparação dos dados

**Projeto:** Futebol e economia · Visualização de Dados · Turma Bosch
**Equipe:** Luan Saviski, Miguel Gradella, Gabriel Wendt

Este notebook carrega os dois datasets (World Cup e World Economic Indicators, ambos da Maven Analytics),
limpa cada um, junta os dois pelo país e pelo ano e salva o resultado em `data/limpos/`.
No final há uma primeira resposta, só com números, para cada pergunta da Etapa 1.

**Recorte:** Copas de 1990 a 2018. O `HDI.csv` começa em 1990, e a tabela de jogos de 2022 não tem placar.

## 0. Configuração

Importamos as bibliotecas e a tabela de correspondência de países que fica em `src/paises.py`.
A função `ler_csv` tenta ler o arquivo em UTF-8 e, se der erro de acentuação, lê em `latin-1`.
Isso foi necessário porque um dos arquivos originais tem caracteres fora do UTF-8.

In [ ]:
import glob
import os
import sys

import numpy as np
import pandas as pd

# permite importar src/paises.py rodando o notebook de dentro de notebooks/
RAIZ = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.append(os.path.join(RAIZ, "src"))
from paises import nome_para_iso3

PASTA_DADOS = os.path.join(RAIZ, "data")
PASTA_LIMPOS = os.path.join(PASTA_DADOS, "limpos")
os.makedirs(PASTA_LIMPOS, exist_ok=True)

ANO_INICIO, ANO_FIM = 1990, 2018
pd.set_option("display.max_columns", 30)


def achar_arquivo(nome):
    """Procura o arquivo em qualquer subpasta de data/ (fora de data/limpos/)."""
    achados = [f for f in glob.glob(os.path.join(PASTA_DADOS, "**", nome), recursive=True)
               if "limpos" not in f]
    if not achados:
        raise FileNotFoundError(f"Não achei {nome} dentro de data/")
    return achados[0]


def ler_csv(nome):
    caminho = achar_arquivo(nome)
    try:
        return pd.read_csv(caminho)
    except UnicodeDecodeError:
        print(f"{nome}: lido com encoding latin-1")
        return pd.read_csv(caminho, encoding="latin-1")

## 1. Carregar

Lemos os três arquivos que respondem às perguntas:
- `world_cup_matches.csv`: um jogo de Copa por linha, com placar
- `world_cups.csv`: uma Copa por linha, com sede e campeão
- `HDI.csv`: indicadores de desenvolvimento humano por país, com uma coluna por ano

In [ ]:
jogos_raw = ler_csv("world_cup_matches.csv")
copas_raw = ler_csv("world_cups.csv")
hdi_raw = ler_csv("HDI.csv")

## 2. Inspecionar

Antes de mexer nos dados, olhamos tamanho, tipos e primeiras linhas de cada tabela.

In [ ]:
for nome, df in [("jogos", jogos_raw), ("copas", copas_raw), ("hdi", hdi_raw)]:
    print(f"{nome}: {df.shape[0]} linhas x {df.shape[1]} colunas")

jogos_raw.info()
jogos_raw.head()

In [ ]:
copas_raw.head()

In [ ]:
hdi_raw.iloc[:5, :8]

## 3. Selecionar e renomear

**Jogos:** ficamos só com as colunas das perguntas e trocamos os nomes para português, sem espaços.
A coluna `Win Conditions` entra porque indica jogos decididos nos pênaltis.

**HDI:** das mais de mil colunas, usamos só `iso3`, `country`, `region` e as séries de
renda nacional bruta per capita (`gnipc_<ano>`, em dólares PPC de 2017) e IDH (`hdi_<ano>`).

In [ ]:
jogos = jogos_raw[["Year", "Stage", "Home Team", "Home Goals", "Away Goals",
                   "Away Team", "Win Conditions"]].rename(columns={
    "Year": "ano",
    "Stage": "fase",
    "Home Team": "time_casa",
    "Home Goals": "gols_casa",
    "Away Goals": "gols_fora",
    "Away Team": "time_fora",
    "Win Conditions": "condicao_vitoria",
})

copas = copas_raw[["Year", "Host Country", "Winner"]].rename(columns={
    "Year": "ano", "Host Country": "sede", "Winner": "campeao"})

colunas_hdi = ["iso3", "country", "region"] + [
    c for c in hdi_raw.columns if c.startswith("gnipc_") or (c.startswith("hdi_") and c[4:].isdigit())]
hdi = hdi_raw[colunas_hdi].rename(columns={"country": "pais_hdi", "region": "regiao"})

print(jogos.columns.tolist())
print(len(colunas_hdi), "colunas do HDI mantidas")

## 4. Transformar o HDI para o formato longo

O `HDI.csv` está no formato **largo**: uma coluna por ano (`gnipc_1990`, `gnipc_1991`...).
Para juntar com os jogos precisamos de uma linha por **país e ano**. Usamos `pd.melt()`
para empilhar as colunas e depois `pivot_table` para separar renda e IDH em duas colunas.

Também removemos as **linhas agregadas** (regiões e grupos de desenvolvimento). Elas não são países
e têm código `iso3` começando com `ZZ`.

In [ ]:
agregadas = hdi["iso3"].astype(str).str.startswith("ZZ") | hdi["iso3"].isna()
print("Linhas agregadas removidas:", agregadas.sum())
hdi = hdi[~agregadas]

hdi_longo = hdi.melt(id_vars=["iso3", "pais_hdi", "regiao"], var_name="indicador_ano", value_name="valor")
hdi_longo["indicador"] = hdi_longo["indicador_ano"].str.rsplit("_", n=1).str[0]
hdi_longo["ano"] = hdi_longo["indicador_ano"].str.rsplit("_", n=1).str[1].astype(int)

hdi_longo = (hdi_longo
             .set_index(["iso3", "pais_hdi", "regiao", "ano", "indicador"])["valor"]
             .unstack("indicador")
             .reset_index()
             .rename(columns={"gnipc": "renda_pc", "hdi": "idh"}))
hdi_longo.columns.name = None
hdi_longo.head()

## 5. Corrigir tipos

Gols e ano precisam ser números inteiros, e renda e IDH números decimais.
`pd.to_numeric(..., errors="coerce")` transforma qualquer texto inválido em `NaN`, que tratamos no passo seguinte.
Nos nomes de seleções removemos espaços extras com `.str.strip()`.

In [ ]:
for col in ["ano", "gols_casa", "gols_fora"]:
    jogos[col] = pd.to_numeric(jogos[col], errors="coerce")
for col in ["time_casa", "time_fora", "fase"]:
    jogos[col] = jogos[col].astype(str).str.strip()

hdi_longo["renda_pc"] = pd.to_numeric(hdi_longo["renda_pc"], errors="coerce")
hdi_longo["idh"] = pd.to_numeric(hdi_longo["idh"], errors="coerce")

jogos.dtypes

## 6. Tratar valores ausentes

Contamos os ausentes antes de decidir.
- **Jogos sem placar** não servem para nenhuma pergunta: são removidos.
- **`condicao_vitoria` vazia** é normal (jogo decidido no tempo normal): preenchemos com texto vazio.
- **Renda ou IDH ausentes** no HDI não são removidos aqui. Eles ficam como `NaN` e saem só das contas que dependem deles,
  para não perder os jogos nas perguntas que não usam economia (como a pergunta 4).

In [ ]:
print("Ausentes nos jogos:")
print(jogos.isna().sum())

antes = len(jogos)
jogos = jogos.dropna(subset=["ano", "gols_casa", "gols_fora"])
jogos["condicao_vitoria"] = jogos["condicao_vitoria"].fillna("")
jogos[["ano", "gols_casa", "gols_fora"]] = jogos[["ano", "gols_casa", "gols_fora"]].astype(int)
print(f"Jogos removidos por falta de placar: {antes - len(jogos)}")

print("\nAusentes no HDI (formato longo):")
print(hdi_longo[["renda_pc", "idh"]].isna().sum())

## 7. Remover duplicados

Um jogo repetido contaria duas vezes nas vitórias e nas médias.

In [ ]:
print("Jogos duplicados:", jogos.duplicated().sum())
jogos = jogos.drop_duplicates()
print("País-ano duplicados no HDI:", hdi_longo.duplicated(subset=["iso3", "ano"]).sum())
hdi_longo = hdi_longo.drop_duplicates(subset=["iso3", "ano"])

## 8. Recortar o período e definir o resultado

Ficamos com as Copas de **1990 a 2018**, o período coberto pelos dois datasets.

**Decisão sobre pênaltis:** usamos o placar do jogo (tempo normal + prorrogação). Um jogo decidido nos pênaltis
conta como **empate**, porque a disputa de pênaltis não mede a força dos times da mesma forma que o jogo.

In [ ]:
jogos = jogos[jogos["ano"].between(ANO_INICIO, ANO_FIM)].copy()
print("Jogos no recorte:", len(jogos), "| Copas:", sorted(jogos["ano"].unique()))

penaltis = jogos["condicao_vitoria"].str.contains("penalt", case=False)
print("Jogos decididos nos pênaltis (contados como empate):", penaltis.sum())

jogos["resultado_casa"] = np.select(
    [jogos["gols_casa"] > jogos["gols_fora"], jogos["gols_casa"] < jogos["gols_fora"]],
    ["vitoria", "derrota"], default="empate")
jogos["resultado_casa"].value_counts()

## 9. Padronizar os países e unir

O merge só encontra chaves **exatamente iguais**. Por isso convertemos cada seleção para o código `iso3`
usando a tabela de `src/paises.py` (Inglaterra → GBR, Alemanha Ocidental → DEU, União Soviética → RUS...).

Depois juntamos a renda e o IDH **do ano da Copa** para o time da casa e para o de fora.

In [ ]:
jogos["iso3_casa"] = jogos["time_casa"].apply(nome_para_iso3)
jogos["iso3_fora"] = jogos["time_fora"].apply(nome_para_iso3)

sem_codigo = sorted(set(jogos.loc[jogos["iso3_casa"].isna(), "time_casa"]) |
                    set(jogos.loc[jogos["iso3_fora"].isna(), "time_fora"]))
print("Seleções sem código iso3:", sem_codigo if sem_codigo else "nenhuma")
# Se aparecer algum nome aqui, adicione-o em src/paises.py e rode de novo.

eco = hdi_longo[["iso3", "ano", "renda_pc", "idh"]]
n_antes = len(jogos)
jogos = (jogos
         .merge(eco.rename(columns={"iso3": "iso3_casa", "renda_pc": "renda_casa", "idh": "idh_casa"}),
                on=["iso3_casa", "ano"], how="left")
         .merge(eco.rename(columns={"iso3": "iso3_fora", "renda_pc": "renda_fora", "idh": "idh_fora"}),
                on=["iso3_fora", "ano"], how="left"))
assert len(jogos) == n_antes, "o merge duplicou linhas"

print(f"Linhas antes e depois do merge: {n_antes} -> {len(jogos)}")
print("Jogos sem renda de algum dos times:", jogos[["renda_casa", "renda_fora"]].isna().any(axis=1).sum())
times_sem_renda = sorted(set(jogos.loc[jogos["renda_casa"].isna(), "time_casa"]) |
                         set(jogos.loc[jogos["renda_fora"].isna(), "time_fora"]))
print("Seleções sem dado de renda em alguma Copa:", times_sem_renda)

## 10. Colunas derivadas

Para as perguntas 1 a 3 cada jogo precisa responder: **o time mais rico venceu, empatou ou perdeu?**
- `diferenca_renda`: renda per capita do mais rico menos a do mais pobre (sempre positiva)
- `resultado_mais_rico`: vitória, empate ou derrota do lado com maior renda

Jogos sem renda de algum dos times ficam com essas colunas vazias.

In [ ]:
casa_mais_rica = jogos["renda_casa"] > jogos["renda_fora"]
tem_renda = jogos[["renda_casa", "renda_fora"]].notna().all(axis=1) & (jogos["renda_casa"] != jogos["renda_fora"])

jogos["diferenca_renda"] = (jogos["renda_casa"] - jogos["renda_fora"]).abs().where(tem_renda)
inverter = {"vitoria": "derrota", "derrota": "vitoria", "empate": "empate"}
jogos["resultado_mais_rico"] = np.where(casa_mais_rica, jogos["resultado_casa"],
                                        jogos["resultado_casa"].map(inverter))
jogos.loc[~tem_renda, "resultado_mais_rico"] = np.nan

jogos[["ano", "time_casa", "time_fora", "gols_casa", "gols_fora",
       "renda_casa", "renda_fora", "diferenca_renda", "resultado_mais_rico"]].head()

**Tabela por seleção.** Para as perguntas 4 e 5 a unidade é a seleção, não o jogo.
Cada jogo vira duas linhas, uma do ponto de vista de cada time.

In [ ]:
lado_casa = pd.DataFrame({
    "ano": jogos["ano"], "selecao": jogos["time_casa"], "iso3": jogos["iso3_casa"],
    "adversario": jogos["time_fora"], "gols_pro": jogos["gols_casa"], "gols_contra": jogos["gols_fora"],
    "resultado": jogos["resultado_casa"], "renda_pc": jogos["renda_casa"], "idh": jogos["idh_casa"]})
lado_fora = pd.DataFrame({
    "ano": jogos["ano"], "selecao": jogos["time_fora"], "iso3": jogos["iso3_fora"],
    "adversario": jogos["time_casa"], "gols_pro": jogos["gols_fora"], "gols_contra": jogos["gols_casa"],
    "resultado": jogos["resultado_casa"].map(inverter), "renda_pc": jogos["renda_fora"], "idh": jogos["idh_fora"]})
selecoes = pd.concat([lado_casa, lado_fora], ignore_index=True)
selecoes["vitoria"] = (selecoes["resultado"] == "vitoria").astype(int)
print(selecoes.shape)
selecoes.head()

## 11. Salvar

Gravamos os resultados limpos para os próximos notebooks (gráficos).

In [ ]:
jogos.to_csv(os.path.join(PASTA_LIMPOS, "jogos_limpos.csv"), index=False)
selecoes.to_csv(os.path.join(PASTA_LIMPOS, "selecoes_por_jogo.csv"), index=False)
hdi_longo.to_csv(os.path.join(PASTA_LIMPOS, "hdi_longo.csv"), index=False)
copas[copas["ano"].between(ANO_INICIO, ANO_FIM)].to_csv(os.path.join(PASTA_LIMPOS, "copas.csv"), index=False)
print(sorted(os.listdir(PASTA_LIMPOS)))

---
# Primeiras estatísticas

Antes dos gráficos, respondemos a cada pergunta da Etapa 1 só com números.
Depois de rodar, **escrevam abaixo de cada resultado 1 ou 2 frases com a conclusão** (célula "Leitura").

In [ ]:
jogos[["gols_casa", "gols_fora", "renda_casa", "renda_fora", "diferenca_renda"]].describe().round(1)

### Pergunta 1 · O país com maior renda per capita vence com mais frequência?

In [ ]:
p1 = jogos["resultado_mais_rico"].value_counts(normalize=True).mul(100).round(1)
n1 = jogos["resultado_mais_rico"].notna().sum()
print(f"Jogos com renda dos dois times: {n1}")
print(p1)
print(f"\nO mais rico venceu {p1.get('vitoria', 0)}% dos jogos e perdeu {p1.get('derrota', 0)}%.")

**Leitura:** _(escrever depois de rodar)_

### Pergunta 2 · A diferença de renda é maior quando o mais rico vence?

In [ ]:
p2 = jogos.groupby("resultado_mais_rico")["diferenca_renda"].describe()[["count", "mean", "50%"]]
p2.columns = ["jogos", "media_US$", "mediana_US$"]
p2.round(0)

**Leitura:** _(compare as medianas: se a de vitória for bem maior, a distância econômica pesa)_

### Pergunta 3 · A vantagem do mais rico mudou de 1990 a 2018?

In [ ]:
p3 = (jogos.dropna(subset=["resultado_mais_rico"])
      .groupby("ano")["resultado_mais_rico"]
      .apply(lambda s: (s == "vitoria").mean() * 100)
      .round(1).rename("% vitórias do mais rico"))
print(p3)
inclinacao = np.polyfit(p3.index, p3.values, 1)[0]
print(f"\nTendência: {inclinacao * 4:+.1f} pontos percentuais por Copa")

**Leitura:** _(escrever depois de rodar)_

### Pergunta 4 · Onde estão os países que mais venceram desde 1990?

Esta pergunta não depende da renda, então usa todos os jogos do recorte. Seleções com o mesmo `iso3`
(Inglaterra e Escócia → GBR) são somadas, porque o mapa mostra países.

In [ ]:
p4 = (selecoes.groupby("iso3")
      .agg(vitorias=("vitoria", "sum"), jogos=("vitoria", "size"))
      .assign(aproveitamento_pct=lambda d: (d["vitorias"] / d["jogos"] * 100).round(1))
      .sort_values("vitorias", ascending=False))
print("Países com pelo menos 1 jogo:", len(p4))
p4.head(10)

**Leitura:** _(escrever depois de rodar)_

### Pergunta 5 · Seleções de países com IDH maior marcam mais gols por jogo?

Unidade: seleção em cada Copa (ex.: Brasil 2002), com o IDH daquele ano e a média de gols por jogo.

In [ ]:
p5 = (selecoes.dropna(subset=["idh"])
      .groupby(["selecao", "ano"])
      .agg(idh=("idh", "first"), gols_por_jogo=("gols_pro", "mean"))
      .reset_index())
r = np.corrcoef(p5["idh"], p5["gols_por_jogo"])[0, 1]
print(f"Seleções-Copa analisadas: {len(p5)}")
print(f"Correlação entre IDH e gols por jogo: {r:.2f}")

**Leitura:** _(de −1 a 1: perto de 0 = sem relação; 0,3 a 0,5 = moderada; acima de 0,5 = forte)_

---
## Decisões de limpeza (resumo para a documentação)

| Decisão | Motivo |
| --- | --- |
| Recorte 1990–2018 | Período em que há placar e dado econômico |
| Leitura em `latin-1` quando necessário | Arquivo com acentos fora do UTF-8 |
| HDI convertido para formato longo | Uma linha por país e ano permite o merge |
| Linhas agregadas do HDI removidas | Regiões e grupos não são países |
| Seleções → `iso3` por tabela própria | Nomes diferentes entre os datasets |
| Inglaterra, Escócia, Gales e Irlanda do Norte → GBR | HDI só tem o Reino Unido |
| Pênaltis contam como empate | O placar do jogo mede melhor a força dos times |
| Renda ausente mantida como `NaN` | Não perder jogos nas perguntas que não usam economia |